# 06 · Tiempos estimados de desplazamiento multimodal

Este notebook calcula los tiempos de desplazamiento tomando como origen los 131 barrios de Madrid hasta los destinos fijados medinate **OpenTripPlanner (OTP)**.

Para tener una aplicacion realista, se ha decidio comparar cuatro modos de transporte:

* **TRANSIT:**: transporte público.
* **WALK:**: a pie.
* **BICYCLE:**: bicicleta.
* **CAR:**: coche.

El objetivo es obtener tiempos comparables para cada barrio, destino y modo, manteniendo dos escenarios diarios: ida por la mañana y vuelta por la tarde.

**NOTA:**: Se guardan los resultados, errores y datos de auditoría para poder comprobar y reproducir el proceso.

**NOTA:** El modo de transporte se considera una preferencia del usuario. La optimización sigue basándose únicamente en precio y tiempo de desplazamiento.

El notebook se ejecuta primero con *TEST_MODE = True* para comprobar que todo funciona correctamente antes de lanzar las 3.144 consultas definitivas.


## 1. Importaciones

In [56]:
from __future__ import annotations

import json
import time
from datetime import date, datetime, time as dt_time
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import pandas as pd
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## 2. Localización automática del proyecto

Puede ejecutarse desde la raíz del proyecto o desde la carpeta *notebooks*.

In [57]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()

    candidates = [start, *start.parents]
    for candidate in candidates:
        zones_file = candidate / "data" / "processed" / "zones_master.csv"
        if zones_file.exists():
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la raíz del proyecto. "
        "Debe existir data/processed/zones_master.csv."
    )


In [58]:
PROJECT_ROOT = find_project_root()

ZONES_PATH = PROJECT_ROOT / "data" / "processed" / "zones_master.csv"
DESTINATIONS_PATH = PROJECT_ROOT / "data" / "reference" / "destinations.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"

# Se USAN nombres nuevos para no mezclar los resultados multimodales con los CSV antiguos que contenían únicamente transporte público.
REQUEST_AUDIT_PATH = OUTPUT_DIR / "transport_route_requests_audit.csv"
ITINERARIES_PATH = OUTPUT_DIR / "transport_route_itineraries.csv"
SUMMARY_PATH = OUTPUT_DIR / "transport_routes_by_neighborhood.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [59]:
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Barrios: {ZONES_PATH}")
print(f"Destinos: {DESTINATIONS_PATH}")

Raíz del proyecto: /home/aclaver/TFG-INFO
Barrios: /home/aclaver/TFG-INFO/data/processed/zones_master.csv
Destinos: /home/aclaver/TFG-INFO/data/reference/destinations.csv


## 3. Configuración del experimento

### Escenarios:

- **Mañana:** llegar al destino como máximo a las 09:00.
- **Tarde:** salir del destino a partir de las 18:00.

**NOTA:** La fecha elegida tiene que estar cubierta por los calendarios GTFS descargados para poder calcular correctamente el transporte público.

### Modos de transporte:

Se calculan cuatro alternativas independientes:

- `TRANSIT`
- `WALK`
- `BICYCLE`
- `CAR`

La unidad experimental pasa a ser: **barrio × destino × escenario × modo de transporte**

Con 131 barrios, 3 destinos, 2 escenarios y 4 modos, la ejecución definitiva contempla **3.144 consultas.**

### Modo de prueba:

**NTOA:** Con *TEST_MODE = True* se usan únicamente tres barrios, manteniendo los tres destinos, los dos escenarios y los cuatro modos. Antes de ejecutar el cálculo definitivo se debe comprobar que los cuatro modos producen respuestas coherentes.

In [60]:
OTP_ENDPOINT = "http://localhost:8080/otp/gtfs/v1"
MADRID_TZ = ZoneInfo("Europe/Madrid")

In [61]:
# Fecha laborable incluida en los GTFS descargados.
SERVICE_DATE = date.fromisoformat("2026-07-29")

In [62]:
# Hay que mantener True durante la validación inicial; solo cambiar a False cuando la prueba multimodal complete las 72 consultas esperadas y supere todas las validaciones.
TEST_MODE = False


In [63]:
TEST_ZONE_KEYS = ["MAD-011", "MAD-104", "MAD-192"]

MAX_ITINERARIES = 3
REQUEST_TIMEOUT_SECONDS = 210
RETRIES = 2
CHECKPOINT_EVERY = 10
RESUME = True

# Permite ejecutar el notebook a partir de los resultados de routing ya calculados, sin requerir una instancia activa de OpenTripPlanner. Debe mantenerse en False cuando se quieran generar o regenerar rutas.
REUSE_EXISTING_RESULTS = True

In [64]:
TRANSPORT_MODES = [
    {
        "transport_mode": "TRANSIT",
        "transport_mode_label": "Transporte público",
        "otp_modes": {
            "transitOnly": True,
        },
    },
    {
        "transport_mode": "WALK",
        "transport_mode_label": "A pie",
        "otp_modes": {
            "direct": ["WALK"],
            "directOnly": True,
        },
    },
    {
        "transport_mode": "BICYCLE",
        "transport_mode_label": "Bicicleta",
        "otp_modes": {
            "direct": ["BICYCLE"],
            "directOnly": True,
        },
    },
    {
        "transport_mode": "CAR",
        "transport_mode_label": "Coche",
        "otp_modes": {
            "direct": ["CAR"],
            "directOnly": True,
        },
    },
]



In [65]:
SCENARIOS = [
    {
        "scenario_id": "morning_arrival_0900",
        "scenario_name": "Llegada al destino a las 09:00",
        "direction": "to_destination",
        "date_time_field": "latestArrival",
        "clock_time": dt_time(9, 0),
    },
    {
        "scenario_id": "evening_departure_1800",
        "scenario_name": "Salida del destino a las 18:00",
        "direction": "from_destination",
        "date_time_field": "earliestDeparture",
        "clock_time": dt_time(18, 0),
    },
]

In [66]:
print("Fecha de servicio:", SERVICE_DATE, SERVICE_DATE.strftime("%A"))
print("Modo de prueba:", TEST_MODE)

print(
    "Modos de transporte:",
    ", ".join(
        mode["transport_mode"]
        for mode in TRANSPORT_MODES
    ),
)

Fecha de servicio: 2026-07-29 Wednesday
Modo de prueba: False
Modos de transporte: TRANSIT, WALK, BICYCLE, CAR


## 4. Lectura y validación de barrios y destinos

In [67]:
zones = pd.read_csv(ZONES_PATH)
destinations = pd.read_csv(DESTINATIONS_PATH)

required_zone_columns = {
    "zone_key",
    "neighborhood_name",
    "district_name",
    "latitude",
    "longitude",
}


required_destination_columns = {
    "destination_id",
    "destination_name",
    "latitude",
    "longitude",
    "active",
}

In [68]:
missing_zone_columns = required_zone_columns - set(zones.columns)
missing_destination_columns = required_destination_columns - set(destinations.columns)

In [69]:
if missing_zone_columns:
    raise ValueError(f"Faltan columnas en zones_master.csv: {sorted(missing_zone_columns)}")

if missing_destination_columns:
    raise ValueError(
        f"Faltan columnas en destinations.csv: {sorted(missing_destination_columns)}"
    )

In [70]:
zones["zone_key"] = zones["zone_key"].astype(str)
zones["latitude"] = pd.to_numeric(zones["latitude"], errors="coerce")
zones["longitude"] = pd.to_numeric(zones["longitude"], errors="coerce")

In [71]:
destinations["destination_id"] = destinations["destination_id"].astype(str)
destinations["latitude"] = pd.to_numeric(destinations["latitude"], errors="coerce")
destinations["longitude"] = pd.to_numeric(destinations["longitude"], errors="coerce")
destinations["active"] = pd.to_numeric(destinations["active"], errors="coerce").fillna(0).astype(int)

In [72]:
active_destinations = destinations.loc[destinations["active"].eq(1)].copy()

if zones["zone_key"].duplicated().any():
    raise ValueError("Existen zone_key duplicados.")

if active_destinations["destination_id"].duplicated().any():
    raise ValueError("Existen destination_id duplicados.")

if zones[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay barrios sin coordenadas.")

if active_destinations[["latitude", "longitude"]].isna().any().any():
    raise ValueError("Hay destinos activos sin coordenadas.")

if len(zones) != 131:
    raise ValueError(f"Se esperaban 131 barrios y se han encontrado {len(zones)}.")

if active_destinations.empty:
    raise ValueError("No hay destinos activos.")


In [73]:
print(f"Barrios: {len(zones)}")
print(f"Destinos activos: {len(active_destinations)}")

display(
    active_destinations[
        ["destination_id", "destination_name", "latitude", "longitude"]
    ]
)

Barrios: 131
Destinos activos: 3


,destination_id,destination_name,latitude,longitude
0,DEST_001,Puerta del Sol,40.416775,-3.703790
1,DEST_002,Nuevos Ministerios,40.446610,-3.692436
2,DEST_003,UC3M Campus de Leganés,40.331755,-3.765697


## 5. Comprobación de OpenTripPlanner

**NOTA:** Docker Desktop debe estar abierto y el contenedor *otp-madrid* debe estar ejecutándose.

In [74]:
HEALTH_QUERY = """
query HealthCheck {
  __typename
}
"""

try:
    health_response = requests.post(
        OTP_ENDPOINT,
        json={
            "query": HEALTH_QUERY,
            "operationName": "HealthCheck",
        },
        headers={
            "Content-Type": "application/json",
            "Accept-Language": "es",
        },
        timeout=30,
    )

    health_response.raise_for_status()
    health_payload = health_response.json()

    if health_payload.get("errors"):
        raise RuntimeError(
            "OTP responde, pero GraphQL devuelve errores:\n"
            + json.dumps(
                health_payload["errors"],
                indent=2,
                ensure_ascii=False,
            )
        )

    print("OpenTripPlanner responde correctamente.")
    print(health_payload)

except requests.exceptions.ReadTimeout as exc:
    raise RuntimeError(
        "OpenTripPlanner está conectado, pero no responde. "
        "Es necesario reiniciar el contenedor otp-madrid."
    ) from exc

except requests.exceptions.ConnectionError as exc:
    raise RuntimeError(
        "No se puede conectar con OpenTripPlanner. "
        "Comprueba que el contenedor otp-madrid está iniciado."
    ) from exc

OpenTripPlanner responde correctamente.
{'data': {'__typename': 'QueryType'}}


## 6. Consulta GraphQL de planificación

**NOTA:** Hay que usar la consulta *planConnection*

El resultado mantiene:

- Duración total.
- Hora de salida y llegada.
- Tiempo andando.
- Distancia andando.
- Tiempo de espera.
- Número de transbordos.
- Modos utilizados.
- Líneas y operadores.

In [75]:
PLAN_QUERY = """
query PlanRoute(
  $origin: PlanLabeledLocationInput!,
  $destination: PlanLabeledLocationInput!,
  $dateTime: PlanDateTimeInput!,
  $first: Int!,
  $modes: PlanModesInput!
) {
  planConnection(
    origin: $origin,
    destination: $destination,
    dateTime: $dateTime,
    first: $first,
    modes: $modes
  ) {
    searchDateTime
    routingErrors {
      code
      description
      inputField
    }
    edges {
      node {
        start
        end
        duration
        generalizedCost
        waitingTime
        walkTime
        walkDistance
        numberOfTransfers
        legs {
          mode
          duration
          distance
          transitLeg
          from {
            name
          }
          to {
            name
          }
          route {
            shortName
            longName
          }
          agency {
            name
          }
        }
      }
    }
  }
}
"""

## 7. Funciones auxiliares

In [76]:
def build_location(label: str, latitude: float, longitude: float) -> dict[str, Any]:
    return {
        "label": str(label),
        "location": {
            "coordinate": {
                "latitude": float(latitude),
                "longitude": float(longitude),
            }
        },
    }


In [77]:
def build_scenario_datetime(
    service_date: date,
    clock_time: dt_time,
) -> str:
    value = datetime.combine(
        service_date,
        clock_time,
        tzinfo=MADRID_TZ,
    )
    return value.isoformat()


def build_request_key(
    service_date: date | str,
    zone_key: str,
    destination_id: str,
    scenario_id: str,
    transport_mode: str,
) -> str:
    """Construye la clave canónica de una consulta OTP."""

    if isinstance(service_date, date):
        service_date_value = service_date.isoformat()
    else:
        service_date_value = str(service_date).strip()

    return (
        f"{service_date_value}|"
        f"{str(zone_key).strip()}|"
        f"{str(destination_id).strip()}|"
        f"{str(scenario_id).strip()}|"
        f"{str(transport_mode).strip().upper()}"
    )


In [78]:
def graphql_request(
    query: str,
    variables: dict[str, Any],
    operation_name: str,
    retries: int = RETRIES,
) -> dict[str, Any]:
    last_exception: Exception | None = None

    for attempt in range(retries + 1):
        try:
            response = requests.post(
                OTP_ENDPOINT,
                json={
                    "query": query,
                    "variables": variables,
                    "operationName": operation_name,
                },
                headers={
                    "Content-Type": "application/json",
                    "OTPTimeout": "180000",
                    "Accept-Language": "es",
                },
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            payload = response.json()

            if payload.get("errors"):
                raise RuntimeError(
                    json.dumps(payload["errors"], ensure_ascii=False)
                )

            return payload

        except (requests.RequestException, ValueError, RuntimeError) as exc:
            last_exception = exc

            if attempt < retries:
                wait_seconds = 2 ** attempt
                print(
                    f"Reintento {attempt + 1}/{retries} "
                    f"tras error: {exc}"
                )
                time.sleep(wait_seconds)

    raise RuntimeError(f"Consulta OTP fallida: {last_exception}")

In [79]:
def parse_itinerary(
    itinerary: dict[str, Any],
    base_record: dict[str, Any],
    rank: int,
) -> dict[str, Any]:
    legs = itinerary.get("legs") or []

    all_modes = [
        leg.get("mode")
        for leg in legs
        if leg.get("mode")
    ]

    transit_modes = [
        leg.get("mode")
        for leg in legs
        if leg.get("transitLeg") and leg.get("mode")
    ]

    leg_distances = [
        leg.get("distance")
        for leg in legs
        if leg.get("distance") is not None
    ]

    routes_used = []
    agencies_used = []

    for leg in legs:
        route = leg.get("route") or {}
        agency = leg.get("agency") or {}

        route_label = route.get("shortName") or route.get("longName")
        if route_label:
            routes_used.append(str(route_label))

        agency_name = agency.get("name")
        if agency_name:
            agencies_used.append(str(agency_name))

    return {
        **base_record,
        "itinerary_rank": rank,
        "start_datetime": itinerary.get("start"),
        "end_datetime": itinerary.get("end"),

        "duration_minutes": (
            itinerary.get("duration") / 60
            if itinerary.get("duration") is not None
            else None
        ),

        "distance_meters": (
            sum(leg_distances)
            if leg_distances
            else None
        ),

        "walk_time_minutes": (
            itinerary.get("walkTime") / 60
            if itinerary.get("walkTime") is not None
            else None
        ),

        "waiting_time_minutes": (
            itinerary.get("waitingTime") / 60
            if itinerary.get("waitingTime") is not None
            else None
        ),

        "walk_distance_meters": itinerary.get("walkDistance"),
        "number_of_transfers": itinerary.get("numberOfTransfers"),
        "generalized_cost": itinerary.get("generalizedCost"),

        "leg_count": len(legs),

        "transit_leg_count": sum(
            bool(leg.get("transitLeg"))
            for leg in legs
        ),

        "all_modes": "|".join(dict.fromkeys(all_modes)),
        "transit_modes": "|".join(dict.fromkeys(transit_modes)),
        "routes_used": "|".join(dict.fromkeys(routes_used)),
        "agencies_used": "|".join(dict.fromkeys(agencies_used)),

        "legs_json": json.dumps(
            legs,
            ensure_ascii=False,
            separators=(",", ":"),
        ),
    }

## 8. Prueba de una única relación en los cuatro modos

Antes de automatizar todas las consultas se prueba una misma combinación barrio–destino–escenario con los cuatro modos de transporte.

Con ello se pretende comprobar que:

- La consulta GraphQL acepta los cuatro modos
- OTP devuelve rutas directas para *WALK*, *BICYCLE* y *CAR*
- *TRANSIT* sigue funcionando como antes
- Los resultados quedan identificados por *transport_mode*

In [80]:
test_zone = zones.loc[
    zones["zone_key"].eq("MAD-192")
].iloc[0]

test_destination = active_destinations.loc[
    active_destinations["destination_id"].eq("DEST_002")
].iloc[0]

test_scenario = SCENARIOS[0]

test_datetime = build_scenario_datetime(
    SERVICE_DATE,
    test_scenario["clock_time"],
)

test_payloads_by_mode: dict[str, dict[str, Any]] = {}

for mode_config in TRANSPORT_MODES:

    transport_mode = mode_config["transport_mode"]

    test_variables = {
        "origin": build_location(
            test_zone["neighborhood_name"],
            test_zone["latitude"],
            test_zone["longitude"],
        ),

        "destination": build_location(
            test_destination["destination_name"],
            test_destination["latitude"],
            test_destination["longitude"],
        ),

        "dateTime": {
            test_scenario["date_time_field"]: test_datetime,
        },

        "first": MAX_ITINERARIES,

        "modes": mode_config["otp_modes"],
    }

    test_payload = graphql_request(
        PLAN_QUERY,
        test_variables,
        "PlanRoute",
    )

    test_plan = test_payload["data"]["planConnection"]

    test_payloads_by_mode[transport_mode] = {
        "edges": test_plan.get("edges") or [],
        "routing_errors": test_plan.get("routingErrors") or [],
    }

In [81]:
print("Fecha/hora de consulta:", test_datetime)

print(
    "Relación de prueba:",
    f"{test_zone['neighborhood_name']} → "
    f"{test_destination['destination_name']}",
)

for transport_mode, result in test_payloads_by_mode.items():

    print(
        f"{transport_mode}: "
        f"{len(result['edges'])} itinerarios; "
        f"{len(result['routing_errors'])} errores de enrutamiento"
    )

Fecha/hora de consulta: 2026-07-29T09:00:00+02:00
Relación de prueba: Valdebernardo → Nuevos Ministerios
TRANSIT: 3 itinerarios; 0 errores de enrutamiento
WALK: 1 itinerarios; 0 errores de enrutamiento
BICYCLE: 1 itinerarios; 0 errores de enrutamiento
CAR: 1 itinerarios; 0 errores de enrutamiento


In [82]:
test_rows = []

for mode_config in TRANSPORT_MODES:

    transport_mode = mode_config["transport_mode"]

    result = test_payloads_by_mode[transport_mode]

    edges = result["edges"]
    routing_errors = result["routing_errors"]

    if not edges:

        test_rows.append(
            {
                "transport_mode": transport_mode,
                "transport_mode_label": mode_config["transport_mode_label"],
                "request_status": "no_route",
                "itineraries_found": 0,
                "duration_minutes": None,
                "distance_meters": None,
                "walk_time_minutes": None,
                "waiting_time_minutes": None,
                "number_of_transfers": None,
                "all_modes": None,
                "transit_modes": None,
                "routing_errors_json": json.dumps(
                    routing_errors,
                    ensure_ascii=False,
                    separators=(",", ":"),
                ),
            }
        )

        continue

    test_base = {
        "request_key": (
            f"TEST|{test_zone['zone_key']}|"
            f"{test_destination['destination_id']}|"
            f"{test_scenario['scenario_id']}|"
            f"{transport_mode}"
        ),

        "zone_key": test_zone["zone_key"],
        "neighborhood_name": test_zone["neighborhood_name"],
        "district_name": test_zone["district_name"],

        "destination_id": test_destination["destination_id"],
        "destination_name": test_destination["destination_name"],

        "scenario_id": test_scenario["scenario_id"],

        "transport_mode": transport_mode,
        "transport_mode_label": mode_config["transport_mode_label"],

        "service_date": SERVICE_DATE.isoformat(),
        "requested_datetime": test_datetime,
    }

    parsed = parse_itinerary(
        edges[0]["node"],
        test_base,
        rank=1,
    )

    test_rows.append(
        {
            "transport_mode": transport_mode,
            "transport_mode_label": mode_config["transport_mode_label"],
            "request_status": "success",
            "itineraries_found": len(edges),

            "duration_minutes": parsed["duration_minutes"],
            "distance_meters": parsed["distance_meters"],

            "walk_time_minutes": parsed["walk_time_minutes"],
            "waiting_time_minutes": parsed["waiting_time_minutes"],
            "number_of_transfers": parsed["number_of_transfers"],

            "all_modes": parsed["all_modes"],
            "transit_modes": parsed["transit_modes"],

            "routing_errors_json": json.dumps(
                routing_errors,
                ensure_ascii=False,
                separators=(",", ":"),
            ),
        }
    )

test_results = pd.DataFrame(test_rows)

In [83]:
display(
    test_results[
        [
            "transport_mode",
            "transport_mode_label",
            "request_status",
            "itineraries_found",
            "duration_minutes",
            "distance_meters",
            "walk_time_minutes",
            "waiting_time_minutes",
            "number_of_transfers",
            "all_modes",
            "transit_modes",
            "routing_errors_json",
        ]
    ]
)

,transport_mode,transport_mode_label,request_status,itineraries_found,duration_minutes,distance_meters,walk_time_minutes,waiting_time_minutes,number_of_transfers,all_modes,transit_modes,routing_errors_json
0,TRANSIT,Transporte público,success,3,115.900000,16056.58,99.166667,6.166667,2,WALK|BUS,BUS,[]
1,WALK,A pie,success,1,140.683333,10572.22,140.683333,0.000000,0,WALK,,[]
2,BICYCLE,Bicicleta,success,1,46.900000,11289.58,0.000000,0.000000,0,BICYCLE,,[]
3,CAR,Coche,success,1,14.833333,12583.39,0.000000,0.000000,0,CAR,,[]


## 9. Selección de barrios para la ejecución

En modo de prueba se utilizan tres barrios.  
En modo definitivo se utilizan los 131.

In [84]:
if TEST_MODE:
    missing_test_keys = sorted(set(TEST_ZONE_KEYS) - set(zones["zone_key"]))
    if missing_test_keys:
        raise ValueError(
            f"Los siguientes barrios de prueba no existen: {missing_test_keys}"
        )

    zones_to_process = (
        zones.loc[zones["zone_key"].isin(TEST_ZONE_KEYS)]
        .sort_values("zone_key")
        .copy()
    )
else:
    zones_to_process = zones.sort_values("zone_key").copy()

expected_requests = (
    len(zones_to_process)
    * len(active_destinations)
    * len(SCENARIOS)
    * len(TRANSPORT_MODES)
)



In [85]:
print(f"Barrios a procesar: {len(zones_to_process)}")
print(f"Destinos: {len(active_destinations)}")
print(f"Escenarios: {len(SCENARIOS)}")
print(f"Modos de transporte: {len(TRANSPORT_MODES)}")
print(f"Consultas previstas: {expected_requests}")

display(
    zones_to_process[
        [
            "zone_key",
            "neighborhood_name",
            "district_name",
        ]
    ]
)

Barrios a procesar: 131
Destinos: 3
Escenarios: 2
Modos de transporte: 4
Consultas previstas: 3144


,zone_key,neighborhood_name,district_name
0,MAD-011,Palacio,Centro
1,MAD-012,Embajadores,Centro
2,MAD-013,Cortes,Centro
3,MAD-014,Justicia,Centro
4,MAD-015,Universidad,Centro
...,...,...,...
126,MAD-211,Alameda de Osuna,Barajas
127,MAD-212,Aeropuerto,Barajas
128,MAD-213,Casco Histórico de Barajas,Barajas
129,MAD-214,Timón,Barajas


## 10. Recuperación de progreso anterior

Si el notebook se interrumpe, puede continuar sin repetir las consultas ya completadas correctamente.

In [86]:
def safe_read_csv(path: Path) -> pd.DataFrame:
    """
    Lee un CSV únicamente si existe y contiene datos.

    La comprobación evita `EmptyDataError` cuando una ejecución
    interrumpida haya creado un fichero vacío.
    """
    if not path.exists():
        return pd.DataFrame()

    if path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def canonicalize_request_keys(
    data: pd.DataFrame,
    *,
    itinerary_table: bool = False,
) -> tuple[pd.DataFrame, int]:
    """
    Reconstruye `request_key` con el formato canónico:

    fecha | barrio | destino | escenario | modo

    También consolida duplicados que puedan proceder de versiones
    anteriores del notebook. De esta forma, el cambio de formato de
    la clave no obliga a borrar resultados ya calculados.
    """

    if data.empty:
        return data.copy(), 0

    result = data.copy()

    required_columns = {
        "service_date",
        "zone_key",
        "destination_id",
        "scenario_id",
        "transport_mode",
    }

    missing_columns = (
        required_columns
        - set(result.columns)
    )

    if missing_columns:
        raise ValueError(
            "No se pueden normalizar los resultados existentes. "
            "Faltan columnas: "
            f"{sorted(missing_columns)}"
        )

    parsed_service_date = pd.to_datetime(
        result["service_date"],
        errors="coerce",
    )

    if parsed_service_date.isna().any():
        raise ValueError(
            "Existen fechas de servicio no válidas en los "
            "resultados anteriores."
        )

    result["service_date"] = (
        parsed_service_date
        .dt.strftime("%Y-%m-%d")
    )

    result["zone_key"] = (
        result["zone_key"]
        .astype(str)
        .str.strip()
    )

    result["destination_id"] = (
        result["destination_id"]
        .astype(str)
        .str.strip()
    )

    result["scenario_id"] = (
        result["scenario_id"]
        .astype(str)
        .str.strip()
    )

    result["transport_mode"] = (
        result["transport_mode"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    result["request_key"] = [
        build_request_key(
            service_date=row.service_date,
            zone_key=row.zone_key,
            destination_id=row.destination_id,
            scenario_id=row.scenario_id,
            transport_mode=row.transport_mode,
        )
        for row in result[
            [
                "service_date",
                "zone_key",
                "destination_id",
                "scenario_id",
                "transport_mode",
            ]
        ].itertuples(index=False)
    ]

    # Si existe una marca temporal, se ordena para conservar
    # la ejecución más reciente al consolidar duplicados.
    result["_original_row_order"] = range(len(result))

    if "calculation_timestamp" in result.columns:
        result["_calculation_timestamp_sort"] = pd.to_datetime(
            result["calculation_timestamp"],
            errors="coerce",
            utc=True,
        )

        result = result.sort_values(
            [
                "_calculation_timestamp_sort",
                "_original_row_order",
            ],
            na_position="first",
        )
    else:
        result = result.sort_values(
            "_original_row_order"
        )

    before = len(result)

    duplicate_subset = ["request_key"]

    if itinerary_table:
        if "itinerary_rank" not in result.columns:
            raise ValueError(
                "El archivo de itinerarios no contiene "
                "`itinerary_rank`."
            )

        duplicate_subset.append(
            "itinerary_rank"
        )

    result = (
        result
        .drop_duplicates(
            subset=duplicate_subset,
            keep="last",
        )
        .drop(
            columns=[
                column
                for column in [
                    "_original_row_order",
                    "_calculation_timestamp_sort",
                ]
                if column in result.columns
            ]
        )
        .reset_index(drop=True)
    )

    removed_duplicates = (
        before - len(result)
    )

    return result, removed_duplicates


if RESUME:
    existing_audit = safe_read_csv(
        REQUEST_AUDIT_PATH
    )

    existing_itineraries = safe_read_csv(
        ITINERARIES_PATH
    )
else:
    existing_audit = pd.DataFrame()
    existing_itineraries = pd.DataFrame()


audit_duplicates_removed = 0
itinerary_duplicates_removed = 0


if not existing_audit.empty:

    (
        existing_audit,
        audit_duplicates_removed,
    ) = canonicalize_request_keys(
        existing_audit,
        itinerary_table=False,
    )

    # Se persiste la migración para que los CSV queden ya
    # en el formato canónico.
    existing_audit.to_csv(
        REQUEST_AUDIT_PATH,
        index=False,
        encoding="utf-8-sig",
    )


if not existing_itineraries.empty:

    (
        existing_itineraries,
        itinerary_duplicates_removed,
    ) = canonicalize_request_keys(
        existing_itineraries,
        itinerary_table=True,
    )

    existing_itineraries.to_csv(
        ITINERARIES_PATH,
        index=False,
        encoding="utf-8-sig",
    )


expected_request_keys_for_run = {
    build_request_key(
        SERVICE_DATE,
        zone_key,
        destination_id,
        scenario["scenario_id"],
        mode["transport_mode"],
    )

    for zone_key
    in zones_to_process["zone_key"]

    for destination_id
    in active_destinations[
        "destination_id"
    ]

    for scenario
    in SCENARIOS

    for mode
    in TRANSPORT_MODES
}


if len(expected_request_keys_for_run) != expected_requests:
    raise ValueError(
        "El número de claves esperadas no coincide "
        "con el número de consultas previstas."
    )


completed_keys: set[str] = set()

if (
    not existing_audit.empty
    and {
        "request_key",
        "request_status",
    }.issubset(existing_audit.columns)
):

    completed_keys = set(
        existing_audit.loc[
            existing_audit[
                "request_status"
            ].isin(
                [
                    "success",
                    "no_route",
                ]
            )
            & existing_audit[
                "request_key"
            ].isin(
                expected_request_keys_for_run
            ),
            "request_key",
        ].astype(str)
    )


pending_requests = (
    expected_requests
    - len(completed_keys)
)


print(
    "Consultas ya completadas y reutilizables "
    "para el experimento actual:",
    len(completed_keys),
)

print(
    "Consultas pendientes:",
    pending_requests,
)


if (
    audit_duplicates_removed
    or itinerary_duplicates_removed
):

    print(
        "Resultados anteriores normalizados: "
        f"{audit_duplicates_removed} filas duplicadas "
        "eliminadas de la auditoría y "
        f"{itinerary_duplicates_removed} itinerarios "
        "duplicados consolidados."
    )


Consultas ya completadas y reutilizables para el experimento actual: 3144
Consultas pendientes: 0


## 11. Ejecución automatizada

In [87]:
new_audit_records: list[dict[str, Any]] = []
new_itinerary_records: list[dict[str, Any]] = []

total_counter = 0
executed_counter = 0


for _, zone in zones_to_process.iterrows():

    for _, destination in active_destinations.iterrows():

        for scenario in SCENARIOS:

            for mode_config in TRANSPORT_MODES:

                transport_mode = mode_config["transport_mode"]
                transport_mode_label = mode_config[
                    "transport_mode_label"
                ]

                total_counter += 1

                # La clave identifica de forma única:
                # fecha + barrio + destino + escenario + modo.
                request_key = build_request_key(
                    SERVICE_DATE,
                    zone["zone_key"],
                    destination["destination_id"],
                    scenario["scenario_id"],
                    transport_mode,
                )

                if request_key in completed_keys:
                    continue

                requested_datetime = build_scenario_datetime(
                    SERVICE_DATE,
                    scenario["clock_time"],
                )

                # -----------------------------------------------------
                # Origen y destino según el sentido del desplazamiento
                # -----------------------------------------------------

                if scenario["direction"] == "to_destination":

                    origin = build_location(
                        zone["neighborhood_name"],
                        zone["latitude"],
                        zone["longitude"],
                    )

                    target = build_location(
                        destination["destination_name"],
                        destination["latitude"],
                        destination["longitude"],
                    )

                elif scenario["direction"] == "from_destination":

                    origin = build_location(
                        destination["destination_name"],
                        destination["latitude"],
                        destination["longitude"],
                    )

                    target = build_location(
                        zone["neighborhood_name"],
                        zone["latitude"],
                        zone["longitude"],
                    )

                else:
                    raise ValueError(
                        "Dirección desconocida: "
                        f"{scenario['direction']}"
                    )

                # -----------------------------------------------------
                # Variables enviadas a OpenTripPlanner
                # -----------------------------------------------------

                variables = {
                    "origin": origin,
                    "destination": target,

                    "dateTime": {
                        scenario["date_time_field"]:
                            requested_datetime,
                    },

                    "first": MAX_ITINERARIES,

                    # Elemento fundamental para el cálculo multimodal.
                    "modes": mode_config["otp_modes"],
                }

                # -----------------------------------------------------
                # Información común de auditoría
                # -----------------------------------------------------

                base_record = {
                    "request_key": request_key,

                    "zone_key": zone["zone_key"],
                    "neighborhood_name":
                        zone["neighborhood_name"],
                    "district_name":
                        zone["district_name"],

                    "origin_latitude":
                        origin["location"]["coordinate"]["latitude"],
                    "origin_longitude":
                        origin["location"]["coordinate"]["longitude"],

                    "destination_id":
                        destination["destination_id"],
                    "destination_name":
                        destination["destination_name"],

                    "target_latitude":
                        target["location"]["coordinate"]["latitude"],
                    "target_longitude":
                        target["location"]["coordinate"]["longitude"],

                    "scenario_id":
                        scenario["scenario_id"],
                    "scenario_name":
                        scenario["scenario_name"],
                    "direction":
                        scenario["direction"],

                    "transport_mode":
                        transport_mode,
                    "transport_mode_label":
                        transport_mode_label,

                    "service_date":
                        SERVICE_DATE.isoformat(),
                    "requested_datetime":
                        requested_datetime,

                    "calculation_timestamp":
                        datetime.now(MADRID_TZ).isoformat(),
                }

                started_at = time.perf_counter()

                # -----------------------------------------------------
                # Consulta OTP
                # -----------------------------------------------------

                try:

                    payload = graphql_request(
                        PLAN_QUERY,
                        variables,
                        "PlanRoute",
                    )

                    plan = payload["data"]["planConnection"]

                    edges = plan.get("edges") or []
                    routing_errors = (
                        plan.get("routingErrors") or []
                    )

                    elapsed_seconds = (
                        time.perf_counter() - started_at
                    )

                    if edges:

                        request_status = "success"

                        for rank, edge in enumerate(
                            edges,
                            start=1,
                        ):
                            new_itinerary_records.append(
                                parse_itinerary(
                                    edge["node"],
                                    base_record,
                                    rank,
                                )
                            )

                    else:
                        request_status = "no_route"

                    new_audit_records.append(
                        {
                            **base_record,

                            "request_status":
                                request_status,

                            "itineraries_found":
                                len(edges),

                            "routing_errors_json":
                                json.dumps(
                                    routing_errors,
                                    ensure_ascii=False,
                                    separators=(",", ":"),
                                ),

                            "request_error":
                                None,

                            "elapsed_seconds":
                                elapsed_seconds,
                        }
                    )

                except Exception as exc:

                    elapsed_seconds = (
                        time.perf_counter() - started_at
                    )

                    new_audit_records.append(
                        {
                            **base_record,

                            "request_status":
                                "error",

                            "itineraries_found":
                                0,

                            "routing_errors_json":
                                "[]",

                            "request_error":
                                str(exc),

                            "elapsed_seconds":
                                elapsed_seconds,
                        }
                    )

                executed_counter += 1

                # -----------------------------------------------------
                # Checkpoint
                # -----------------------------------------------------

                if (
                    executed_counter % CHECKPOINT_EVERY == 0
                    or executed_counter == pending_requests
                ):

                    combined_audit = pd.concat(
                        [
                            existing_audit,
                            pd.DataFrame(
                                new_audit_records
                            ),
                        ],
                        ignore_index=True,
                    ).drop_duplicates(
                        subset=["request_key"],
                        keep="last",
                    )

                    combined_itineraries = pd.concat(
                        [
                            existing_itineraries,
                            pd.DataFrame(
                                new_itinerary_records
                            ),
                        ],
                        ignore_index=True,
                    )

                    if not combined_itineraries.empty:

                        combined_itineraries = (
                            combined_itineraries
                            .drop_duplicates(
                                subset=[
                                    "request_key",
                                    "itinerary_rank",
                                ],
                                keep="last",
                            )
                        )

                    combined_audit.to_csv(
                        REQUEST_AUDIT_PATH,
                        index=False,
                        encoding="utf-8-sig",
                    )

                    # Solo se escribe el fichero si hay itinerarios válidos.
                    if not combined_itineraries.empty:
                        combined_itineraries.to_csv(
                            ITINERARIES_PATH,
                            index=False,
                            encoding="utf-8-sig",
                        )

                    print(
                        "Progreso guardado: "
                        f"{total_counter}/"
                        f"{expected_requests} combinaciones; "
                        f"{executed_counter} "
                        "nuevas consultas."
                    )


In [88]:
# Guardado final obligatorio, aunque el número de consultas nuevas no sea múltiplo de CHECKPOINT_EVERY.

combined_audit = pd.concat(
    [
        existing_audit,
        pd.DataFrame(new_audit_records),
    ],
    ignore_index=True,
).drop_duplicates(
    subset=["request_key"],
    keep="last",
)


combined_itineraries = pd.concat(
    [
        existing_itineraries,
        pd.DataFrame(new_itinerary_records),
    ],
    ignore_index=True,
)


if not combined_itineraries.empty:

    combined_itineraries = (
        combined_itineraries
        .drop_duplicates(
            subset=[
                "request_key",
                "itinerary_rank",
            ],
            keep="last",
        )
    )


combined_audit.to_csv(
    REQUEST_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)


if not combined_itineraries.empty:

    combined_itineraries.to_csv(
        ITINERARIES_PATH,
        index=False,
        encoding="utf-8-sig",
    )

In [89]:
current_run_audit_count = int(
    combined_audit[
        "request_key"
    ].isin(
        expected_request_keys_for_run
    ).sum()
)


if combined_itineraries.empty:
    current_run_itinerary_count = 0
else:
    current_run_itinerary_count = int(
        combined_itineraries[
            "request_key"
        ].isin(
            expected_request_keys_for_run
        ).sum()
    )


print(
    "Guardado final completado para el experimento actual: "
    f"{current_run_audit_count} consultas auditadas y "
    f"{current_run_itinerary_count} itinerarios."
)

print(
    "Nuevas consultas ejecutadas en esta sesión:",
    executed_counter,
)

print("Ejecución terminada.")


Guardado final completado para el experimento actual: 3144 consultas auditadas y 4423 itinerarios.
Nuevas consultas ejecutadas en esta sesión: 0
Ejecución terminada.


## 12. Lectura de resultados y auditoría

Una vez terminadas las consultas se recuperan los archivos generados y se restringen al experimento actual.

La auditoría conserva una fila por combinación: **fecha de servicio × barrio × destino × escenario × modo de transporte**

Así se puede distinguir entre:

- consultas correctas (*success*)
- consultas para las que OTP no ha encontrado itinerario (*no_route*)
- errores técnicos (*error*)

**NOTA:** Los casos *no_route* se conservan explícitamente y no se sustituyen por tiempos artificiales.


In [90]:
request_audit = safe_read_csv(
    REQUEST_AUDIT_PATH
)

itineraries = safe_read_csv(
    ITINERARIES_PATH
)


if request_audit.empty:
    raise ValueError(
        "El archivo de auditoría está vacío. "
        "Ejecuta primero el cálculo de rutas."
    )

if itineraries.empty:
    raise ValueError(
        "El archivo de itinerarios está vacío. "
        "No se puede construir el resumen de rutas."
    )


# ---------------------------------------------------------
# Validación mínima de estructura
# ---------------------------------------------------------

required_audit_columns = {
    "request_key",
    "zone_key",
    "destination_id",
    "scenario_id",
    "transport_mode",
    "transport_mode_label",
    "service_date",
    "request_status",
    "routing_errors_json",
    "request_error",
}

required_itinerary_columns = {
    "request_key",
    "zone_key",
    "destination_id",
    "scenario_id",
    "transport_mode",
    "transport_mode_label",
    "itinerary_rank",
    "duration_minutes",
}


missing_audit_columns = (
    required_audit_columns
    - set(request_audit.columns)
)

missing_itinerary_columns = (
    required_itinerary_columns
    - set(itineraries.columns)
)


if missing_audit_columns:
    raise ValueError(
        "Faltan columnas en la auditoría: "
        f"{sorted(missing_audit_columns)}"
    )

if missing_itinerary_columns:
    raise ValueError(
        "Faltan columnas en los itinerarios: "
        f"{sorted(missing_itinerary_columns)}"
    )


# ---------------------------------------------------------
# Restricción al experimento actual
# ---------------------------------------------------------

current_zone_keys = set(
    zones_to_process["zone_key"]
)

current_destination_ids = set(
    active_destinations["destination_id"]
)

current_scenarios = {
    scenario["scenario_id"]
    for scenario in SCENARIOS
}

current_modes = {
    mode["transport_mode"]
    for mode in TRANSPORT_MODES
}


current_audit = request_audit.loc[
    request_audit["zone_key"].isin(
        current_zone_keys
    )
    & request_audit["destination_id"].isin(
        current_destination_ids
    )
    & request_audit["scenario_id"].isin(
        current_scenarios
    )
    & request_audit["transport_mode"].isin(
        current_modes
    )
    & request_audit["service_date"].astype(str).eq(
        SERVICE_DATE.isoformat()
    )
].copy()


current_itineraries = itineraries.loc[
    itineraries["zone_key"].isin(
        current_zone_keys
    )
    & itineraries["destination_id"].isin(
        current_destination_ids
    )
    & itineraries["scenario_id"].isin(
        current_scenarios
    )
    & itineraries["transport_mode"].isin(
        current_modes
    )
    & itineraries["service_date"].astype(str).eq(
        SERVICE_DATE.isoformat()
    )
].copy()


print(
    "Consultas recuperadas para el experimento actual:",
    len(current_audit),
)

print(
    "Itinerarios recuperados:",
    len(current_itineraries),
)

Consultas recuperadas para el experimento actual: 3144
Itinerarios recuperados: 4423


In [91]:
# Estado global de las consultas

status_summary = (
    current_audit[
        "request_status"
    ]
    .value_counts(dropna=False)
    .rename_axis("request_status")
    .reset_index(name="consultas")
)


print("Estado global de las consultas:")
display(status_summary)


# Estado desglosado por modo de transporte

status_by_mode = (
    current_audit
    .groupby(
        [
            "transport_mode",
            "transport_mode_label",
            "request_status",
        ],
        dropna=False,
    )
    .size()
    .rename("consultas")
    .reset_index()
    .sort_values(
        [
            "transport_mode",
            "request_status",
        ]
    )
)


print("\nEstado de las consultas por modo:")
display(status_by_mode)

# Itinerarios almacenados por modo

itinerary_summary_by_mode = (
    current_itineraries
    .groupby(
        [
            "transport_mode",
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        consultas_con_itinerario=(
            "request_key",
            "nunique",
        ),
        itinerarios_almacenados=(
            "request_key",
            "size",
        ),
    )
)


itinerary_summary_by_mode[
    "alternativas_medias_por_consulta"
] = (
    itinerary_summary_by_mode[
        "itinerarios_almacenados"
    ]
    / itinerary_summary_by_mode[
        "consultas_con_itinerario"
    ]
).round(2)


print("\nItinerarios almacenados por modo:")
display(itinerary_summary_by_mode)

Estado global de las consultas:


,request_status,consultas
0,success,2933
1,no_route,211



Estado de las consultas por modo:


,transport_mode,transport_mode_label,request_status,consultas
0,BICYCLE,Bicicleta,no_route,6
1,BICYCLE,Bicicleta,success,780
2,CAR,Coche,no_route,54
3,CAR,Coche,success,732
4,TRANSIT,Transporte público,no_route,37
5,TRANSIT,Transporte público,success,749
6,WALK,A pie,no_route,114
7,WALK,A pie,success,672



Itinerarios almacenados por modo:


,transport_mode,transport_mode_label,consultas_con_itinerario,itinerarios_almacenados,alternativas_medias_por_consulta
0,BICYCLE,Bicicleta,780,780,1.00
1,CAR,Coche,732,732,1.00
2,TRANSIT,Transporte público,749,2239,2.99
3,WALK,A pie,672,672,1.00


### 12.1. Resultados de la ejecución

Las tablas anteriores muestran de forma automática la ejecución actual, por lo que no se fijan manualmente cifras que puedan quedar desactualizadas al cambiar entre el modo de prueba y la ejecución definitiva.

La primera tabla permite comprobar el estado global de las consultas. La segunda separa los resultados por modo de transporte, lo que es especialmente importante en el nuevo diseño multimodal.

Finalmente, se muestra cuántos itinerarios se han almacenado para cada modo. **OpenTripPlanner* puede devolver varias alternativas para una misma consulta; todas ellas se conservan para mantener la trazabilidad, aunque posteriormente se seleccionará *itinerary_rank = 1* como alternativa principal.

In [92]:
error_requests = current_audit.loc[
    current_audit[
        "request_status"
    ].eq("error"),
    [
        "zone_key",
        "neighborhood_name",
        "destination_name",
        "scenario_id",
        "transport_mode",
        "transport_mode_label",
        "request_error",
    ],
].copy()


no_route_requests = current_audit.loc[
    current_audit[
        "request_status"
    ].eq("no_route"),
    [
        "zone_key",
        "neighborhood_name",
        "destination_name",
        "scenario_id",
        "transport_mode",
        "transport_mode_label",
        "routing_errors_json",
    ],
].copy()

In [93]:
print(
    "Consultas con error técnico:",
    len(error_requests),
)

print(
    "Consultas sin itinerario:",
    len(no_route_requests),
)


if not error_requests.empty:

    print(
        "\nErrores técnicos detectados:"
    )

    display(
        error_requests.sort_values(
            [
                "transport_mode",
                "destination_name",
                "neighborhood_name",
                "scenario_id",
            ]
        )
    )


if not no_route_requests.empty:

    print(
        "\nConsultas sin itinerario por modo:"
    )

    display(
        no_route_requests[
            "transport_mode"
        ]
        .value_counts()
        .rename_axis(
            "transport_mode"
        )
        .reset_index(
            name="consultas_sin_itinerario"
        )
    )

Consultas con error técnico: 0
Consultas sin itinerario: 211

Consultas sin itinerario por modo:


,transport_mode,consultas_sin_itinerario
0,WALK,114
1,CAR,54
2,TRANSIT,37
3,BICYCLE,6


In [94]:
no_route_audit = current_audit.loc[
    current_audit[
        "request_status"
    ].eq("no_route")
].copy()


def extract_routing_codes(
    value: Any,
) -> list[str]:
    """
    Extrae los códigos devueltos por OTP
    en routing_errors_json.
    """
    try:
        errors = json.loads(value)
    except (
        TypeError,
        json.JSONDecodeError,
    ):
        return ["ERROR_DE_LECTURA"]

    if not errors:
        return ["SIN_CODIGO"]

    return [
        str(
            error.get(
                "code",
                "DESCONOCIDO",
            )
        )
        for error in errors
    ]


no_route_code_records = []


for _, row in no_route_audit.iterrows():

    codes = extract_routing_codes(
        row["routing_errors_json"]
    )

    for code in codes:

        no_route_code_records.append(
            {
                "transport_mode":
                    row["transport_mode"],

                "transport_mode_label":
                    row["transport_mode_label"],

                "codigo":
                    code,
            }
        )


if no_route_code_records:

    no_route_codes = pd.DataFrame(
        no_route_code_records
    )

    resumen_no_route = (
        no_route_codes
        .groupby(
            [
                "transport_mode",
                "transport_mode_label",
                "codigo",
            ],
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "numero_consultas"
            }
        )
    )

    totals_by_mode = (
        resumen_no_route
        .groupby(
            "transport_mode"
        )[
            "numero_consultas"
        ]
        .transform("sum")
    )

    resumen_no_route[
        "porcentaje_modo"
    ] = (
        100
        * resumen_no_route[
            "numero_consultas"
        ]
        / totals_by_mode
    ).round(2)

    resumen_no_route = (
        resumen_no_route
        .sort_values(
            [
                "transport_mode",
                "numero_consultas",
            ],
            ascending=[
                True,
                False,
            ],
        )
        .reset_index(drop=True)
    )

else:

    resumen_no_route = pd.DataFrame(
        columns=[
            "transport_mode",
            "transport_mode_label",
            "codigo",
            "numero_consultas",
            "porcentaje_modo",
        ]
    )

In [95]:
if resumen_no_route.empty:

    print(
        "No existen consultas sin itinerario."
    )

else:

    print(
        "Motivos de las consultas sin itinerario:"
    )

    display(resumen_no_route)

Motivos de las consultas sin itinerario:


,transport_mode,transport_mode_label,codigo,numero_consultas,porcentaje_modo
0,BICYCLE,Bicicleta,NO_DIRECT_MODE_CONNECTION,6,100.00
1,CAR,Coche,NO_DIRECT_MODE_CONNECTION,54,100.00
2,TRANSIT,Transporte público,WALKING_BETTER_THAN_TRANSIT,23,62.16
3,TRANSIT,Transporte público,NO_STOPS_IN_RANGE,12,32.43
4,TRANSIT,Transporte público,NO_TRANSIT_CONNECTION_IN_SEARCH_WINDOW,2,5.41
5,WALK,A pie,NO_DIRECT_MODE_CONNECTION,114,100.00


In [96]:
if not no_route_audit.empty:

    no_route_detail = (
        no_route_audit[
            [
                "zone_key",
                "neighborhood_name",
                "destination_name",
                "scenario_id",
                "transport_mode",
                "transport_mode_label",
                "routing_errors_json",
            ]
        ]
        .sort_values(
            [
                "transport_mode",
                "destination_name",
                "neighborhood_name",
                "scenario_id",
            ]
        )
        .reset_index(drop=True)
    )

    display(no_route_detail)

,zone_key,neighborhood_name,destination_name,scenario_id,transport_mode,transport_mode_label,routing_errors_json
0,MAD-159,Costillares,Nuevos Ministerios,evening_departure_1800,BICYCLE,Bicicleta,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
1,MAD-159,Costillares,Nuevos Ministerios,morning_arrival_0900,BICYCLE,Bicicleta,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
2,MAD-159,Costillares,Puerta del Sol,evening_departure_1800,BICYCLE,Bicicleta,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
3,MAD-159,Costillares,Puerta del Sol,morning_arrival_0900,BICYCLE,Bicicleta,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
4,MAD-159,Costillares,UC3M Campus de Leganés,evening_departure_1800,BICYCLE,Bicicleta,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
...,...,...,...,...,...,...,...
206,MAD-193,Valderrivas,UC3M Campus de Leganés,morning_arrival_0900,WALK,A pie,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
207,MAD-094,Valdezarza,UC3M Campus de Leganés,evening_departure_1800,WALK,A pie,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
208,MAD-094,Valdezarza,UC3M Campus de Leganés,morning_arrival_0900,WALK,A pie,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."
209,MAD-086,Valverde,UC3M Campus de Leganés,evening_departure_1800,WALK,A pie,"[{""code"":""NO_DIRECT_MODE_CONNECTION"",""description"":""No route found for the requested direct mode. The origin and des..."


### 12.2. Consultas sin itinerario

Una respuesta *no_route* no se interpreta automáticamente como la imposibilidad absoluta de realizar el desplazamiento.

El significado debe analizarse conjuntamente con:

- El modo de transporte solicitado;
- El código devuelto por OpenTripPlanner;
- La coordenada utilizada para representar el barrio;
- El destino y la franja horaria.

En particular, algunos códigos son específicos del transporte público y no deben extrapolarse a los modos directos.

Los casos sin itinerario se mantienen en la auditoría y posteriormente quedan identificados como alternativas no disponibles. No se sustituyen por cero, medias ni tiempos imputados, ya que hacerlo alteraría artificialmente la accesibilidad de la zona.

## 13. Construcción del itinerario principal

OpenTripPlanner puede devolver varias alternativas para una misma consulta. Para construir el dataset principal se selecciona *itinerary_rank = 1*, es decir, la primera alternativa devuelta por el motor de routing.

La unidad de observación continúa siendo: **barrio × destino × escenario × modo de transporte**

El resto de alternativas no se eliminaN y permanecen disponibles en *transport_route_itineraries.csv*, lo que permite conservar la trazabilidad completa del proceso.

In [97]:
best_itineraries = (
    current_itineraries.loc[
        current_itineraries["itinerary_rank"].eq(1)
    ]
    .copy()
)

# Se conserva una única alternativa principal por barrio, destino, escenario y modo de transporte.
best_columns = [
    "zone_key",
    "neighborhood_name",
    "district_name",
    "destination_id",
    "destination_name",
    "scenario_id",

    # Dimensión multimodal
    "transport_mode",
    "transport_mode_label",

    "service_date",
    "requested_datetime",
    "start_datetime",
    "end_datetime",

    # Características principales del itinerario
    "duration_minutes",
    "distance_meters",
    "walk_time_minutes",
    "waiting_time_minutes",
    "walk_distance_meters",
    "number_of_transfers",

    # Información sobre los modos y servicios utilizados
    "all_modes",
    "transit_modes",
    "routes_used",
    "agencies_used",
]

best_itineraries = best_itineraries[best_columns]

# Una consulta queda identificada de forma única por: fecha + barrio + destino + escenario + modo de transporte.
PRIMARY_ITINERARY_KEY = [
    "service_date",
    "zone_key",
    "destination_id",
    "scenario_id",
    "transport_mode",
]

if best_itineraries.duplicated(
    subset=PRIMARY_ITINERARY_KEY
).any():

    duplicated = best_itineraries.loc[
        best_itineraries.duplicated(
            subset=PRIMARY_ITINERARY_KEY,
            keep=False,
        )
    ].sort_values(PRIMARY_ITINERARY_KEY)

    display(duplicated)

    raise ValueError(
        "Hay más de un itinerario principal para una misma "
        "combinación barrio-destino-escenario-modo."
    )

print(
    "Itinerarios principales:",
    len(best_itineraries),
)

print(
    "Modos presentes:",
    sorted(
        best_itineraries["transport_mode"]
        .dropna()
        .unique()
    ),
)

display(best_itineraries.head(12))


Itinerarios principales: 2933
Modos presentes: ['BICYCLE', 'CAR', 'TRANSIT', 'WALK']


,zone_key,neighborhood_name,district_name,destination_id,destination_name,scenario_id,transport_mode,transport_mode_label,service_date,requested_datetime,start_datetime,end_datetime,duration_minutes,distance_meters,walk_time_minutes,waiting_time_minutes,walk_distance_meters,number_of_transfers,all_modes,transit_modes,routes_used,agencies_used
0,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,morning_arrival_0900,WALK,A pie,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:42:29+02:00,2026-07-29T09:00:00+02:00,17.516667,1203.25,17.516667,0.0,1203.25,0,WALK,NaN,NaN,NaN
1,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,morning_arrival_0900,BICYCLE,Bicicleta,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:51:56+02:00,2026-07-29T09:00:00+02:00,8.066667,1216.02,0.000000,0.0,0.00,0,BICYCLE,NaN,NaN,NaN
2,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,morning_arrival_0900,CAR,Coche,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:51:46+02:00,2026-07-29T09:00:00+02:00,8.233333,3518.59,0.000000,0.0,0.00,0,CAR,NaN,NaN,NaN
3,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,evening_departure_1800,WALK,A pie,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-29T18:00:00+02:00,2026-07-29T18:17:24+02:00,17.400000,1203.25,17.400000,0.0,1203.25,0,WALK,NaN,NaN,NaN
4,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,evening_departure_1800,BICYCLE,Bicicleta,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-29T18:00:00+02:00,2026-07-29T18:07:40+02:00,7.666667,1203.25,0.000000,0.0,0.00,0,BICYCLE,NaN,NaN,NaN
5,MAD-011,Palacio,Centro,DEST_001,Puerta del Sol,evening_departure_1800,CAR,Coche,2026-07-29,2026-07-29T18:00:00+02:00,2026-07-29T18:00:00+02:00,2026-07-29T18:07:59+02:00,7.983333,2981.15,0.000000,0.0,0.00,0,CAR,NaN,NaN,NaN
6,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,morning_arrival_0900,TRANSIT,Transporte público,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T07:51:00+02:00,2026-07-29T08:51:33+02:00,60.550000,7509.85,10.633333,13.0,707.85,1,WALK|BUS,BUS,C1,Información oficial: Consorcio Regional de Transportes de Madrid
9,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,morning_arrival_0900,WALK,A pie,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T07:48:06+02:00,2026-07-29T09:00:00+02:00,71.900000,5225.34,71.900000,0.0,5225.34,0,WALK,NaN,NaN,NaN
10,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,morning_arrival_0900,BICYCLE,Bicicleta,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:29:14+02:00,2026-07-29T09:00:00+02:00,30.766667,5676.50,0.000000,0.0,0.00,0,BICYCLE,NaN,NaN,NaN
11,MAD-011,Palacio,Centro,DEST_002,Nuevos Ministerios,morning_arrival_0900,CAR,Coche,2026-07-29,2026-07-29T09:00:00+02:00,2026-07-29T08:48:05+02:00,2026-07-29T09:00:00+02:00,11.916667,5991.43,0.000000,0.0,0.00,0,CAR,NaN,NaN,NaN


## 14. Construcción del desplazamiento diario

Para cada combinación barrio–destino–modo se combinan los dos escenarios diarios:

- ida por la mañana;
- vuelta por la tarde.

**NOTA::** La tabla se construye tomando como base la auditoría y no únicamente los itinerarios encontrados. De esta forma se conservan también las combinaciones para las que OTP no devuelve una ruta.

Una alternativa se clasifica como:

- *complete_route*: existen ida y vuelta
- *partial_route*: únicamente existe uno de los dos trayectos
- *no_route*: OTP no encuentra ninguno de los dos
- *echnical_error*: al menos una consulta ha terminado con error técnico

*commute_daily_minutes* solo se calcula cuando existen correctamente los dos desplazamientos. No se imputan tiempos inexistentes.

In [98]:
value_columns = [
    "duration_minutes",
    "distance_meters",
    "walk_time_minutes",
    "waiting_time_minutes",
    "walk_distance_meters",
    "number_of_transfers",
]


DAILY_ROUTE_KEY = [
    "zone_key",
    "neighborhood_name",
    "district_name",
    "destination_id",
    "destination_name",
    "transport_mode",
    "transport_mode_label",
    "service_date",
]


scenario_definitions = [
    (
        "morning_arrival_0900",
        "morning",
    ),
    (
        "evening_departure_1800",
        "evening",
    ),
]


status_parts = []
itinerary_parts = []


for scenario_id, prefix in scenario_definitions:

    # Estado de todas las consultas

    status_part = current_audit.loc[
        current_audit[
            "scenario_id"
        ].eq(scenario_id),
        [
            *DAILY_ROUTE_KEY,
            "requested_datetime",
            "request_status",
            "routing_errors_json",
            "request_error",
        ],
    ].copy()


    status_part = status_part.rename(
        columns={
            "requested_datetime":
                f"{prefix}_requested_datetime",

            "request_status":
                f"{prefix}_route_status",

            "routing_errors_json":
                f"{prefix}_routing_errors_json",

            "request_error":
                f"{prefix}_request_error",
        }
    )


    if status_part.duplicated(
        subset=DAILY_ROUTE_KEY
    ).any():

        raise ValueError(
            "La auditoría contiene más de una "
            f"consulta para el escenario {scenario_id} "
            "y la misma combinación barrio-destino-modo."
        )


    status_parts.append(status_part)


    # Métricas del itinerario principal, cuando existe

    itinerary_part = best_itineraries.loc[
        best_itineraries[
            "scenario_id"
        ].eq(scenario_id)
    ].copy()


    rename_map = {
        column:
            f"{prefix}_{column}"
        for column in value_columns
    }


    rename_map.update(
        {
            "start_datetime":
                f"{prefix}_start_datetime",

            "end_datetime":
                f"{prefix}_end_datetime",

            "all_modes":
                f"{prefix}_all_modes",

            "transit_modes":
                f"{prefix}_transit_modes",

            "routes_used":
                f"{prefix}_routes_used",

            "agencies_used":
                f"{prefix}_agencies_used",
        }
    )


    itinerary_keep_columns = [
        *DAILY_ROUTE_KEY,

        "start_datetime",
        "end_datetime",

        "all_modes",
        "transit_modes",
        "routes_used",
        "agencies_used",

        *value_columns,
    ]


    itinerary_part = (
        itinerary_part[
            itinerary_keep_columns
        ]
        .rename(
            columns=rename_map
        )
    )


    itinerary_parts.append(
        itinerary_part
    )


# Base completa: una fila por barrio-destino-modo

daily_summary = status_parts[0].merge(
    status_parts[1],
    on=DAILY_ROUTE_KEY,
    how="outer",
    validate="one_to_one",
)

In [99]:
# Incorporación de las métricas de los itinerarios

for itinerary_part in itinerary_parts:

    daily_summary = daily_summary.merge(
        itinerary_part,
        on=DAILY_ROUTE_KEY,
        how="left",
        validate="one_to_one",
    )

In [100]:
# Clasificación del estado diario

morning_success = (
    daily_summary[
        "morning_route_status"
    ].eq("success")
)

evening_success = (
    daily_summary[
        "evening_route_status"
    ].eq("success")
)

morning_no_route = (
    daily_summary[
        "morning_route_status"
    ].eq("no_route")
)

evening_no_route = (
    daily_summary[
        "evening_route_status"
    ].eq("no_route")
)

has_technical_error = (
    daily_summary[
        [
            "morning_route_status",
            "evening_route_status",
        ]
    ]
    .eq("error")
    .any(axis=1)
)


daily_summary[
    "route_status"
] = "incomplete"


daily_summary.loc[
    morning_no_route
    & evening_no_route,
    "route_status",
] = "no_route"


daily_summary.loc[
    morning_success
    ^ evening_success,
    "route_status",
] = "partial_route"


daily_summary.loc[
    morning_success
    & evening_success,
    "route_status",
] = "complete_route"


daily_summary.loc[
    has_technical_error,
    "route_status",
] = "technical_error"


daily_summary[
    "route_available"
] = daily_summary[
    "route_status"
].eq("complete_route")


daily_summary[
    "no_route_scenarios"
] = (
    daily_summary[
        [
            "morning_route_status",
            "evening_route_status",
        ]
    ]
    .eq("no_route")
    .sum(axis=1)
)

In [101]:
# Variables diarias. min_count=2 obliga a disponer de los dos trayectos

daily_summary[
    "commute_daily_minutes"
] = daily_summary[
    [
        "morning_duration_minutes",
        "evening_duration_minutes",
    ]
].sum(
    axis=1,
    min_count=2,
)


daily_summary[
    "commute_daily_distance_meters"
] = daily_summary[
    [
        "morning_distance_meters",
        "evening_distance_meters",
    ]
].sum(
    axis=1,
    min_count=2,
)


daily_summary[
    "commute_daily_walk_minutes"
] = daily_summary[
    [
        "morning_walk_time_minutes",
        "evening_walk_time_minutes",
    ]
].sum(
    axis=1,
    min_count=2,
)


daily_summary[
    "commute_daily_waiting_minutes"
] = daily_summary[
    [
        "morning_waiting_time_minutes",
        "evening_waiting_time_minutes",
    ]
].sum(
    axis=1,
    min_count=2,
)


daily_summary[
    "commute_daily_walk_distance_meters"
] = daily_summary[
    [
        "morning_walk_distance_meters",
        "evening_walk_distance_meters",
    ]
].sum(
    axis=1,
    min_count=2,
)


daily_summary[
    "commute_daily_transfers"
] = daily_summary[
    [
        "morning_number_of_transfers",
        "evening_number_of_transfers",
    ]
].sum(
    axis=1,
    min_count=2,
)

In [102]:
# Orden

daily_summary = (
    daily_summary
    .sort_values(
        [
            "destination_id",
            "zone_key",
            "transport_mode",
        ]
    )
    .reset_index(drop=True)
)

In [103]:
# Guardado

daily_summary.to_csv(
    SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

In [104]:
print(
    "Resumen multimodal guardado en:",
    SUMMARY_PATH,
)

print(
    "Filas del resumen diario:",
    len(daily_summary),
)

print(
    "Modos disponibles:",
    sorted(
        daily_summary[
            "transport_mode"
        ]
        .dropna()
        .unique()
    ),
)


print(
    "\nEstado de las combinaciones diarias:"
)

display(
    daily_summary[
        "route_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "route_status"
    )
    .reset_index(
        name="combinaciones"
    )
)


display(
    daily_summary[
        [
            "zone_key",
            "neighborhood_name",
            "destination_name",
            "transport_mode",
            "transport_mode_label",

            "morning_route_status",
            "evening_route_status",
            "route_status",

            "morning_duration_minutes",
            "evening_duration_minutes",
            "commute_daily_minutes",

            "commute_daily_distance_meters",
        ]
    ].head(30)
)

Resumen multimodal guardado en: /home/aclaver/TFG-INFO/data/processed/transport_routes_by_neighborhood.csv
Filas del resumen diario: 1572
Modos disponibles: ['BICYCLE', 'CAR', 'TRANSIT', 'WALK']

Estado de las combinaciones diarias:


,route_status,combinaciones
0,complete_route,1462
1,no_route,101
2,partial_route,9


,zone_key,neighborhood_name,destination_name,transport_mode,transport_mode_label,morning_route_status,evening_route_status,route_status,morning_duration_minutes,evening_duration_minutes,commute_daily_minutes,commute_daily_distance_meters
0,MAD-011,Palacio,Puerta del Sol,BICYCLE,Bicicleta,success,success,complete_route,8.066667,7.666667,15.733333,2419.27
1,MAD-011,Palacio,Puerta del Sol,CAR,Coche,success,success,complete_route,8.233333,7.983333,16.216667,6499.74
2,MAD-011,Palacio,Puerta del Sol,TRANSIT,Transporte público,no_route,no_route,no_route,NaN,NaN,NaN,NaN
3,MAD-011,Palacio,Puerta del Sol,WALK,A pie,success,success,complete_route,17.516667,17.400000,34.916667,2406.50
4,MAD-012,Embajadores,Puerta del Sol,BICYCLE,Bicicleta,success,success,complete_route,13.216667,9.966667,23.183333,3328.04
5,MAD-012,Embajadores,Puerta del Sol,CAR,Coche,success,success,complete_route,10.116667,11.100000,21.216667,8370.66
6,MAD-012,Embajadores,Puerta del Sol,TRANSIT,Transporte público,no_route,no_route,no_route,NaN,NaN,NaN,NaN
7,MAD-012,Embajadores,Puerta del Sol,WALK,A pie,success,success,complete_route,13.500000,13.433333,26.933333,2032.41
8,MAD-013,Cortes,Puerta del Sol,BICYCLE,Bicicleta,success,success,complete_route,6.600000,9.433333,16.033333,2205.60
9,MAD-013,Cortes,Puerta del Sol,CAR,Coche,success,success,complete_route,6.816667,6.766667,13.583333,5018.76


### 14.1. Cobertura de los desplazamientos diarios

La incorporación del modo de transporte modifica la unidad de análisis.

En la ejecución definitiva se esperan:

***Total de rutas calculadas = 131 (barrios) x 3 (destinos) x 4 (modos de transporte) = 1.572 rutas***

combinaciones barrio–destino–modo.

Cada combinación requiere dos consultas independientes *ida y vuelta*, por lo que la ejecución definitiva contempla: ***1.572 x 2 = 3.144*** consultas a OpenTripPlanner.

**NOTA:** El resumen diario conserva todas las combinaciones auditadas, incluidas aquellas sin itinerario. El tiempo diario únicamente se considera válido cuando existen tanto la ida como la vuelta. De esta forma, cuando  no se consigue una ruta resultante queda representada explícitamente mediante el estado de la alternativa y no mediante tiempos artificiales.

## 15. Validaciones finales

In [105]:
expected_daily_combinations = (
    len(zones_to_process)
    * len(active_destinations)
    * len(TRANSPORT_MODES)
)


observed_daily_combinations = (
    len(daily_summary)
)


route_status_counts = (
    daily_summary[
        "route_status"
    ]
    .value_counts(
        dropna=False
    )
)


complete_routes = int(
    route_status_counts.get(
        "complete_route",
        0,
    )
)

partial_routes = int(
    route_status_counts.get(
        "partial_route",
        0,
    )
)

no_routes = int(
    route_status_counts.get(
        "no_route",
        0,
    )
)

technical_error_routes = int(
    route_status_counts.get(
        "technical_error",
        0,
    )
)

other_routes = int(
    observed_daily_combinations
    - complete_routes
    - partial_routes
    - no_routes
    - technical_error_routes
)


print(
    "Combinaciones barrio-destino-modo esperadas:",
    expected_daily_combinations,
)

print(
    "Combinaciones presentes en el resumen:",
    observed_daily_combinations,
)

print(
    "Rutas completas:",
    complete_routes,
)

print(
    "Rutas parciales:",
    partial_routes,
)

print(
    "Sin ruta:",
    no_routes,
)

print(
    "Con error técnico:",
    technical_error_routes,
)

print(
    "Otros estados:",
    other_routes,
)


if (
    observed_daily_combinations
    != expected_daily_combinations
):
    raise ValueError(
        "El resumen diario no contiene todas las "
        "combinaciones barrio-destino-modo esperadas."
    )



Combinaciones barrio-destino-modo esperadas: 1572
Combinaciones presentes en el resumen: 1572
Rutas completas: 1462
Rutas parciales: 9
Sin ruta: 101
Con error técnico: 0
Otros estados: 0


In [106]:

# Cobertura por modo

coverage_by_mode = (
    daily_summary
    .groupby(
        [
            "transport_mode",
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        combinaciones=(
            "zone_key",
            "size",
        ),

        rutas_completas=(
            "route_available",
            "sum",
        ),
    )
)


coverage_by_mode[
    "cobertura_completa_pct"
] = (
    100
    * coverage_by_mode[
        "rutas_completas"
    ]
    / coverage_by_mode[
        "combinaciones"
    ]
).round(2)


print("\nCobertura por modo:")
display(coverage_by_mode)


Cobertura por modo:


,transport_mode,transport_mode_label,combinaciones,rutas_completas,cobertura_completa_pct
0,BICYCLE,Bicicleta,393,390,99.24
1,CAR,Coche,393,363,92.37
2,TRANSIT,Transporte público,393,373,94.91
3,WALK,A pie,393,336,85.50


In [107]:
expected_request_keys = set(
    expected_request_keys_for_run
)



observed_request_keys = set(
    current_audit[
        "request_key"
    ].astype(str)
)


missing_request_keys = sorted(
    expected_request_keys
    - observed_request_keys
)


unexpected_request_keys = sorted(
    observed_request_keys
    - expected_request_keys
)


duplicate_audit_keys = int(
    current_audit[
        "request_key"
    ]
    .duplicated()
    .sum()
)


duplicate_itinerary_keys = int(
    current_itineraries
    .duplicated(
        subset=[
            "request_key",
            "itinerary_rank",
        ]
    )
    .sum()
)


successful_requests = int(
    current_audit[
        "request_status"
    ]
    .eq("success")
    .sum()
)


failed_requests = int(
    current_audit[
        "request_status"
    ]
    .eq("error")
    .sum()
)


no_route_count = int(
    current_audit[
        "request_status"
    ]
    .eq("no_route")
    .sum()
)


invalid_durations = int(
    (
        pd.to_numeric(
            current_itineraries[
                "duration_minutes"
            ],
            errors="coerce",
        )
        <= 0
    )
    .sum()
)


success_request_keys = set(
    current_audit.loc[
        current_audit[
            "request_status"
        ].eq("success"),
        "request_key",
    ].astype(str)
)


itinerary_request_keys = set(
    current_itineraries[
        "request_key"
    ].astype(str)
)


orphan_itinerary_keys = sorted(
    itinerary_request_keys
    - success_request_keys
)


print(
    "Consultas esperadas:",
    len(expected_request_keys),
)

print(
    "Consultas auditadas:",
    len(observed_request_keys),
)

print(
    "Consultas ausentes:",
    len(missing_request_keys),
)

print(
    "Consultas inesperadas:",
    len(unexpected_request_keys),
)

print(
    "Claves duplicadas en auditoría:",
    duplicate_audit_keys,
)

print(
    "Itinerarios duplicados:",
    duplicate_itinerary_keys,
)

print(
    "Duraciones no positivas:",
    invalid_durations,
)

print(
    "Itinerarios sin consulta success asociada:",
    len(orphan_itinerary_keys),
)

print(
    "Consultas correctas:",
    successful_requests,
)

print(
    "Consultas sin ruta:",
    no_route_count,
)

print(
    "Errores técnicos:",
    failed_requests,
)




Consultas esperadas: 3144
Consultas auditadas: 3144
Consultas ausentes: 0
Consultas inesperadas: 0
Claves duplicadas en auditoría: 0
Itinerarios duplicados: 0
Duraciones no positivas: 0
Itinerarios sin consulta success asociada: 0
Consultas correctas: 2933
Consultas sin ruta: 211
Errores técnicos: 0


In [108]:
# Validaciones estrictas

if missing_request_keys:

    print(
        "Primeras consultas ausentes:"
    )

    print(
        missing_request_keys[:10]
    )

    raise ValueError(
        "Existen consultas esperadas "
        "que no aparecen en la auditoría."
    )


if unexpected_request_keys:

    print(
        "Primeras consultas inesperadas:"
    )

    print(
        unexpected_request_keys[:10]
    )

    raise ValueError(
        "La auditoría contiene consultas "
        "ajenas al experimento actual."
    )


if duplicate_audit_keys:

    raise ValueError(
        "La auditoría contiene "
        "request_key duplicadas."
    )


if duplicate_itinerary_keys:

    raise ValueError(
        "Existen itinerarios duplicados "
        "para una misma request_key y rank."
    )


if invalid_durations:

    raise ValueError(
        "Existen itinerarios con "
        "duración menor o igual que cero."
    )


if orphan_itinerary_keys:

    raise ValueError(
        "Existen itinerarios asociados "
        "a consultas que no figuran como success."
    )


if failed_requests:

    raise ValueError(
        "La ejecución contiene errores técnicos. "
        "Deben resolverse antes de continuar."
    )


print(
    "\nTodas las validaciones estructurales "
    "se han superado correctamente."
)


if TEST_MODE:

    print(
        "\nPRUEBA MULTIMODAL COMPLETADA. "
        "Si los resultados son plausibles, "
        "puede realizarse la ejecución definitiva."
    )

else:

    print(
        "\nEJECUCIÓN MULTIMODAL DEFINITIVA "
        "COMPLETADA CORRECTAMENTE."
    )



Todas las validaciones estructurales se han superado correctamente.

EJECUCIÓN MULTIMODAL DEFINITIVA COMPLETADA CORRECTAMENTE.


In [109]:
complete_daily_routes = (
    daily_summary.loc[
        daily_summary[
            "route_status"
        ].eq("complete_route")
    ]
    .copy()
)


destination_mode_summary = (
    complete_daily_routes
    .groupby(
        [
            "destination_id",
            "destination_name",
            "transport_mode",
            "transport_mode_label",
        ],
        as_index=False,
    )
    .agg(
        barrios_con_ruta_completa=(
            "zone_key",
            "nunique",
        ),

        tiempo_diario_medio=(
            "commute_daily_minutes",
            "mean",
        ),

        tiempo_diario_mediano=(
            "commute_daily_minutes",
            "median",
        ),

        tiempo_diario_minimo=(
            "commute_daily_minutes",
            "min",
        ),

        tiempo_diario_maximo=(
            "commute_daily_minutes",
            "max",
        ),

        distancia_diaria_media_m=(
            "commute_daily_distance_meters",
            "mean",
        ),
    )
)


destination_mode_summary[
    "cobertura_barrios_pct"
] = (
    100
    * destination_mode_summary[
        "barrios_con_ruta_completa"
    ]
    / len(zones_to_process)
)


destination_mode_summary[
    "distancia_diaria_media_km"
] = (
    destination_mode_summary[
        "distancia_diaria_media_m"
    ]
    / 1000
)


destination_mode_summary = (
    destination_mode_summary.drop(
        columns=[
            "distancia_diaria_media_m"
        ]
    )
)


numeric_columns = (
    destination_mode_summary
    .select_dtypes(
        include="number"
    )
    .columns
)


destination_mode_summary[
    numeric_columns
] = (
    destination_mode_summary[
        numeric_columns
    ]
    .round(2)
)


print(
    "Resumen de accesibilidad "
    "por destino y modo:"
)

display(
    destination_mode_summary
    .sort_values(
        [
            "destination_id",
            "transport_mode",
        ]
    )
    .reset_index(drop=True)
)




Resumen de accesibilidad por destino y modo:


,destination_id,destination_name,transport_mode,transport_mode_label,barrios_con_ruta_completa,tiempo_diario_medio,tiempo_diario_mediano,tiempo_diario_minimo,tiempo_diario_maximo,cobertura_barrios_pct,distancia_diaria_media_km
0,DEST_001,Puerta del Sol,BICYCLE,Bicicleta,130,74.42,69.47,3.70,186.65,99.24,16.34
1,DEST_001,Puerta del Sol,CAR,Coche,121,25.48,25.17,9.22,63.85,92.37,16.43
2,DEST_001,Puerta del Sol,TRANSIT,Transporte público,120,119.25,117.88,56.37,244.52,91.60,15.92
3,DEST_001,Puerta del Sol,WALK,A pie,128,177.06,168.92,3.70,435.82,97.71,13.12
4,DEST_002,Nuevos Ministerios,BICYCLE,Bicicleta,130,77.20,76.58,10.50,164.62,99.24,17.04
5,DEST_002,Nuevos Ministerios,CAR,Coche,121,24.43,25.08,5.98,65.22,92.37,17.46
6,DEST_002,Nuevos Ministerios,TRANSIT,Transporte público,125,121.50,120.80,44.97,275.78,95.42,17.25
7,DEST_002,Nuevos Ministerios,WALK,A pie,129,195.32,192.80,21.03,465.92,98.47,14.43
8,DEST_003,UC3M Campus de Leganés,BICYCLE,Bicicleta,130,169.53,174.85,50.48,319.57,99.24,42.67
9,DEST_003,UC3M Campus de Leganés,CAR,Coche,121,38.05,38.37,16.02,76.82,92.37,36.93


In [110]:
# Indicadores específicos del transporte público

transit_complete = (
    complete_daily_routes.loc[
        complete_daily_routes[
            "transport_mode"
        ].eq("TRANSIT")
    ]
    .copy()
)


if not transit_complete.empty:

    transit_quality_summary = (
        transit_complete
        .groupby(
            [
                "destination_id",
                "destination_name",
            ],
            as_index=False,
        )
        .agg(
            caminata_diaria_media_min=(
                "commute_daily_walk_minutes",
                "mean",
            ),

            espera_diaria_media_min=(
                "commute_daily_waiting_minutes",
                "mean",
            ),

            transbordos_diarios_medios=(
                "commute_daily_transfers",
                "mean",
            ),
        )
    )


    transit_numeric_columns = (
        transit_quality_summary
        .select_dtypes(
            include="number"
        )
        .columns
    )


    transit_quality_summary[
        transit_numeric_columns
    ] = (
        transit_quality_summary[
            transit_numeric_columns
        ]
        .round(2)
    )


    print(
        "\nIndicadores específicos "
        "del transporte público:"
    )

    display(
        transit_quality_summary
    )


Indicadores específicos del transporte público:


,destination_id,destination_name,caminata_diaria_media_min,espera_diaria_media_min,transbordos_diarios_medios
0,DEST_001,Puerta del Sol,39.77,22.17,0.83
1,DEST_002,Nuevos Ministerios,35.79,23.82,1.48
2,DEST_003,UC3M Campus de Leganés,40.08,36.17,3.94


## 15.1. Interpretación de los tiempos de desplazamiento

- El tiempo de desplazamiento no es fijo para cada barrio, ya que depende tanto del destino como del modo de transporte. Por eso, cada alternativa se define como barrio + destino + modo de transporte.

- En transporte público también influyen factores como la espera, la caminata y los transbordos, mientras que en coche, bicicleta o andando importan sobre todo la duración y la distancia.

- El modo de transporte no se trata como un objetivo independiente en la optimización. El problema seguirá centrado en el equilibrio entre coste de vivienda y tiempo de desplazamiento, utilizando el modo de transporte que seleccione el usuario.

## 16. Archivos generados

*transport_route_requests_audit.csv*: Contiene una fila por solicitud realizada a OpenTripPlanner.

La clave lógica de cada consulta está formada por: **fecha de servicio + barrio + destino + escenario + modo de transporte**

El archivo permite auditar:

- modo de transporte solicitado;
- estado de la consulta;
- número de itinerarios encontrados;
- errores de routing;
- errores técnicos;
- tiempo empleado por la petición;
- fecha de servicio;
- instante de cálculo.

*transport_route_itineraries.csv*: Contiene una fila por itinerario devuelto por OTP y conserva hasta tres alternativas por consulta.

Entre otras variables incluye:

- duración;
- distancia;
- tiempo andando;
- tiempo de espera;
- transbordos;
- modos utilizados;
- líneas y operadores;
- detalle de los distintos tramos.

El campo *itinerary_rank* permite identificar la alternativa principal y conservar simultáneamente el resto para auditoría.

*transport_routes_by_neighborhood.csv*: Es el resumen principal que utilizará el notebook 07.

Contiene una fila por: **barrio × destino × modo de transporte**

Integra:

- estado del trayecto de mañana;
- estado del trayecto de tarde;
- duración de ambos desplazamientos;
- tiempo diario total;
- distancia diaria;
- variables de caminata;
- tiempo de espera;
- transbordos;
- rutas y operadores;
- estado global de disponibilidad de la alternativa.

Las combinaciones sin ruta también permanecen en el archivo, pero sus tiempos no se imputan.


## 16.1. Limitaciones del análisis

1. Los resultados tienen algunas limitaciones que hay que tener en cuenta. Cada barrio se representa con una única coordenada, por lo que no se reflejan las diferencias que puede haber dentro de una misma zona. Además, las rutas se calculan para un día laborable y unos horarios concretos, así que los tiempos pueden cambiar en otros momentos.

2. En transporte público se utilizan los horarios incluidos en GTFS, sin tener en cuenta retrasos o incidencias puntuales. En coche tampoco se usa tráfico en tiempo real, por lo que los tiempos son una estimación. Las rutas andando y en bicicleta dependen de la información disponible en *OpenStreetMap*.

3. Por último, un resultado *no_route* significa que OTP no encontró una ruta para esa combinación concreta, no que sea imposible desplazarse desde todo el barrio.

## 17. Ejecución definitiva

La ejecución de prueba debe completarse antes de calcular todas las combinaciones.

Con *TEST_MODE = True* se esperan: ***3 x 3 x 4 x 2 = 72 consultas***

La ejecución definitiva utiliza: ***131 (barrios) x 3 (destinos) x 4 (modos) x 2 (escenarios) = 3.144 consultas***

Una vez superadas todas las validaciones:

1. Cambiar *TEST_MODE = True* por *TEST_MODE = False*
2. Reiniciar el kernel.
3. Comprobar que Docker Desktop y el contenedor *otp-madrid* están activos.
4. Ejecutar el notebook completo.
5. Mantener *RESUME = True* para poder reanudar la ejecución en caso de interrupción.
6. Verificar que las validaciones finales indican cero consultas ausentes, cero claves duplicadas y cero errores técnicos.

**NOTA:** Los resultados ya calculados correctamente pueden reutilizarse mediante una *request_key* canónica que incluye la fecha de servicio, evitando repetir consultas idénticas y evitando mezclar ejecuciones de fechas diferentes.
